In [14]:
# # !pip install rosbags
# !pip install open3d

In [15]:
from pathlib import Path
import pandas as pd

from rosbags.highlevel import AnyReader
from rosbags.typesys import Stores, get_typestore

In [16]:
bag_path = Path("../real_episode_0016_1783402673")
typestore = get_typestore(Stores.ROS2_HUMBLE)

rows = []
timestamps = {}
first_msg = {}
joint_names = {}

with AnyReader([bag_path], default_typestore=typestore) as reader:

    connections = reader.connections

    for conn, ts, raw in reader.messages():
        timestamps.setdefault(conn.topic, []).append(ts)

        if conn.topic not in first_msg:
            msg = reader.deserialize(raw, conn.msgtype)
            first_msg[conn.topic] = msg

            if conn.msgtype == "sensor_msgs/msg/JointState":
                joint_names[conn.topic] = list(msg.name)

    for conn in connections:
        ts = timestamps.get(conn.topic, [])

        if len(ts) < 2:
            freq = 0.0
        else:
            duration = (ts[-1] - ts[0]) / 1e9
            freq = (len(ts) - 1) / duration

        msg = first_msg[conn.topic]

        if conn.msgtype == "sensor_msgs/msg/Image":
            shape = f"{msg.height} x {msg.width}"

        elif conn.msgtype == "sensor_msgs/msg/CameraInfo":
            shape = f"{msg.height} x {msg.width}"

        elif conn.msgtype == "sensor_msgs/msg/JointState":
            shape = f"{len(msg.name)} joints"

        elif conn.msgtype == "tf2_msgs/msg/TFMessage":
            shape = f"{len(msg.transforms)} transforms"

        else:
            shape = "-"

        rows.append({
            "Topic": conn.topic,
            "Type": conn.msgtype,
            "Shape": shape,
            "Messages": conn.msgcount,
            "Frequency (Hz)": round(freq, 2),
        })

df = pd.DataFrame(rows)

print("\n=== Topic Summary ===\n")
print(df.to_string(index=False))



=== Topic Summary ===

                              Topic                       Type        Shape  Messages  Frequency (Hz)
                 /xarm/joint_states sensor_msgs/msg/JointState     7 joints       426          152.13
   /camera/camera/color/camera_info sensor_msgs/msg/CameraInfo    480 x 640       164           57.85
     /camera/camera/color/image_raw      sensor_msgs/msg/Image    480 x 640       164           58.04
   /camera/camera/depth/camera_info sensor_msgs/msg/CameraInfo    480 x 640       164           57.86
/camera/camera/depth/image_rect_raw      sensor_msgs/msg/Image    480 x 640       164           58.08
                         /tf_static     tf2_msgs/msg/TFMessage 4 transforms         3            2.02
                      /joint_states sensor_msgs/msg/JointState    13 joints        38           10.00


In [17]:
print("\n=== Joint Names ===")

for topic, names in joint_names.items():
    print(f"\n{topic} ({len(names)} joints)")
    for i, name in enumerate(names):
        print(f"  {i:2d}: {name}")


=== Joint Names ===

/joint_states (13 joints)
   0: joint1
   1: joint2
   2: joint3
   3: joint4
   4: joint5
   5: joint6
   6: joint7
   7: drive_joint
   8: left_finger_joint
   9: left_inner_knuckle_joint
  10: right_outer_knuckle_joint
  11: right_finger_joint
  12: right_inner_knuckle_joint

/xarm/joint_states (7 joints)
   0: joint1
   1: joint2
   2: joint3
   3: joint4
   4: joint5
   5: joint6
   6: joint7


In [18]:
print("\n=== Camera Info ===\n")

for topic in [
    "/camera/camera/color/camera_info",
    "/camera/camera/depth/camera_info",
]:
    msg = first_msg[topic]

    print(f"{topic}")
    print(f"  Resolution : {msg.width} x {msg.height}")
    print(f"  Distortion : {msg.distortion_model}")

    print(f"  K (intrinsics):")
    print(f"    fx = {msg.k[0]:.3f}")
    print(f"    fy = {msg.k[4]:.3f}")
    print(f"    cx = {msg.k[2]:.3f}")
    print(f"    cy = {msg.k[5]:.3f}")

    print(f"  Distortion coeffs:")
    print(f"    {list(msg.d)}")

    print()


=== Camera Info ===

/camera/camera/color/camera_info
  Resolution : 640 x 480
  Distortion : plumb_bob
  K (intrinsics):
    fx = 386.120
    fy = 385.152
    cx = 329.546
    cy = 244.972
  Distortion coeffs:
    [np.float64(-0.05551962926983833), np.float64(0.06743203103542328), np.float64(-0.00011596848344197497), np.float64(0.00016109277203213423), np.float64(-0.02195647917687893)]

/camera/camera/depth/camera_info
  Resolution : 640 x 480
  Distortion : plumb_bob
  K (intrinsics):
    fx = 391.274
    fy = 391.274
    cx = 328.905
    cy = 242.808
  Distortion coeffs:
    [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]



In [20]:
import numpy as np
import rerun as rr
import open3d as o3d

from rosbags.highlevel import AnyReader

# ------------------------------------------------------------------
# Camera intrinsics
# ------------------------------------------------------------------

cam = first_msg["/camera/camera/depth/camera_info"]

intrinsic = o3d.camera.PinholeCameraIntrinsic(
    width=cam.width,
    height=cam.height,
    fx=cam.k[0],
    fy=cam.k[4],
    cx=cam.k[2],
    cy=cam.k[5],
)

rr.init("depth_episode", spawn=True)

frame = 0

with AnyReader([bag_path], default_typestore=typestore) as reader:
    for conn, ts, raw in reader.messages():

        if conn.topic != "/camera/camera/depth/image_rect_raw":
            continue

        msg = reader.deserialize(raw, conn.msgtype)

        # ------------------------------------------------------------
        # Decode depth image
        # ------------------------------------------------------------

        if msg.encoding == "16UC1":
            depth = np.frombuffer(
                msg.data,
                dtype=np.uint16,
            ).reshape(msg.height, msg.width).astype(np.float32)

            depth /= 1000.0

        elif msg.encoding == "32FC1":
            depth = np.frombuffer(
                msg.data,
                dtype=np.float32,
            ).reshape(msg.height, msg.width)

        else:
            continue

        # ------------------------------------------------------------
        # Depth -> Point Cloud
        # ------------------------------------------------------------

        pcd = o3d.geometry.PointCloud.create_from_depth_image(
            o3d.geometry.Image(depth),
            intrinsic,
        )

        pts = np.asarray(pcd.points)

        # Remove invalid points
        pts = pts[np.isfinite(pts).all(axis=1)]

        rr.set_time("frame", sequence=frame)

        rr.log(
            "world/point_cloud",
            rr.Points3D(
                positions=pts,
                colors=pts[:, 2],   # Color by depth (Z)
                radii=0.003,
            ),
        )

        frame += 1

print(f"Logged {frame} frames.")

2026-07-07T10:00:50.325762Z  INFO re_perf_telemetry::telemetry: Telemetry initialized enabled=false service=rerun trace_mode="off" traces=off logs=off metrics=off tracy=false
2026-07-07T10:00:50.333695Z  INFO winit::platform_impl::linux::x11::window: Guessed window scale factor: 1
2026-07-07T10:00:50.531594Z  WARN egui_wgpu::winit: Transparent window was requested, but the active wgpu surface does not support a `CompositeAlphaMode` with transparency.
2026-07-07T10:00:50.573714Z  INFO re_grpc_server: Listening for gRPC connections on 0.0.0.0:9876. Connect by running `rerun --connect rerun+http://127.0.0.1:9876/proxy`


Logged 164 frames.


2026-07-07T10:04:58.385745Z ERROR re_grpc_client::write: Write messages call failed: transport error
